*The harness behind the six checks, what it refuses to tell you, and how it was tested.*

**Six offline checks will tell you whether your change broke your agent. They will not tell you whether your measurement is worth believing.**

That is a separate question and it is the harder one. An offline test can be reproducible, hold out seats, report an interval and still be wrong by fifty-nine points, which is what ours did: **72% on 98 held-out seats, and then 8 wins in 60 real ladder games.** Nothing about it looked broken from the inside, and it stayed in service for a week.

**This notebook is the harness that replaced it and the reasoning that shaped it.** Four gates that can stop a run, four measures that report only what their sample supports, a declared sample requirement for each so an underpowered result cannot be quietly read as a null one, and a self-test that has to pass before anything it produces is published. Along the way: the four families every retracted measurement here belongs to, what you are actually maximising in a two-player duel, and the two places the harness refuses to give you a number at all.

*If you want the checks rather than the argument, they are in [Six checks before you waste a submission](https://www.kaggle.com/code/destbreso/six-checks-before-you-waste-a-submission).*

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path
from scipy import stats


def find(name):
    """Locate an input, wherever this notebook is running.

    EVERY input this notebook uses comes from the attached dataset. Nothing is
    pasted into a cell, so forking it and re-running gives you the same figures
    from the same declared source rather than somebody else's numbers frozen
    into a string literal.
    """
    for root in ("/kaggle/input", ".", "..", "data"):
        r = Path(root)
        if r.exists():
            for p in r.rglob(name):
                return p
    return None


def load_input(name):
    p = find(name)
    if p is None:
        raise FileNotFoundError(
            f"{name} not found. Attach the dataset "
            f"'destbreso/kaggriculture-benchmark-matchups' with Add Input, "
            f"on the right of the editor.")
    return json.loads(Path(p).read_text())

plt.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 130,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22, "grid.linewidth": 0.6,
    "font.size": 9.5, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelsize": 9.5, "legend.frameon": False,
    # Thin and soft throughout. Heavy strokes and big vertices fight the data.
    "lines.linewidth": 1.1, "lines.markersize": 4.5,
    "lines.solid_capstyle": "round", "lines.dash_capstyle": "round",
    "axes.linewidth": 0.7, "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "patch.linewidth": 0.7,
})
INK, TEAL, AMBER, GREY, RED = "#0F172A", "#0F766E", "#B45309", "#94A3B8", "#B91C1C"

## The shape of it: four gates, then four measures

Order matters, because each gate can invalidate everything after it. A failure stops the run rather than being noted and worked around.

| | gate | what it refuses | why it exists |
|---|---|---|---|
| 0 | **ENGINE** | a harness that no longer reproduces recorded games | the same seed pays one public agent 28,370 on one engine build and 9,002 on another |
| 1 | **PARITY** | a control that is not bit-identical | otherwise a change and a bug are the same observation |
| 2 | **SAFETY** | an agent that crashes or exceeds the turn budget | a ladder crash is silent and forfeits |
| 3 | **POPULATION** | a pool that is undiverse, or one you built from your own wins | a win rate over near-duplicates is one game repeated |
| 4 | **SYMMETRY** | data with a seat-dependent fault | the only gate that can fail while your agent is perfect |

| | measure | what it reports |
|---|---|---|
| A | **PAIRED** | the difference against the baseline on the same seed, seat and opponent |
| B | **OBJECTIVE** | `Pr[win] = Φ(μ/σ)`, where the sample supports it and not otherwise |
| C | **SPLIT** | screen and confirm on disjoint halves, and the drop between them |
| D | **RANKING** | a strength ordering, gated by a transitivity check that currently refuses |

The harness itself travels with the dataset, so the cell below is the real thing running rather than a transcript of it. Four of the five gates replay a game and therefore need the competition engine and an agent of yours; the rest is arithmetic on results, and that half runs here on the published benchmark.

In [ ]:
import sys
sys.path.insert(0, str(find("instrument.py").parent))
import instrument as I

print("gates and measures declared:", ", ".join(I.SPEC))
print()
print("what each one says it needs, which is the half that is usually implicit:")
for name in ("POPULATION", "SYMMETRY", "PAIRED"):
    s = I.SPEC[name]
    print(f"  {name:<12} at least {s['min_n']:<5} {s['needs'][:58]}")

In [ ]:
# Feed it the published benchmark. This adapter is the whole interface: if your
# own results can be shaped like this, every check below runs on them unchanged.
B = pd.read_parquet(find("matchups_top.parquet"))
pool = [{"episode_id": r.episode_id, "seed": int(r.seed),
         "our_seat": int(r.your_seat), "opp_seat": int(r.opponent_seat),
         "opponent": r.opponent_team,
         "our_bank": float(r.recorded_bank_yours),
         "their_bank": float(r.recorded_bank_opponent),
         "we_won": bool(r.recorded_bank_yours > r.recorded_bank_opponent),
         "script": json.loads(r.opponent_actions)}
        for r in B.itertuples()]
print(f"{len(pool)} matchups from the strong half of the field")

In [ ]:
# POPULATION clusters every pair of action streams, which is quadratic, so this
# runs it on a slice. The full pool takes a few minutes rather than seconds.
SLICE = 150
pop = I.gate_population(None, pool[:SLICE])
sym = I.gate_symmetry(pool)

# Effective n and the already-won share mean the same thing on any pool, so they
# are what this prints. The gate also estimates field coverage and unseen
# richness; both are defined on a pool collected AS IT CAME and this one is
# stratified, so they belong to a different kind of pool than this.
print(f"POPULATION on {SLICE} matchups")
print(f"  nominal {pop['nominal']}, EFFECTIVE {pop['effective']}")
print(f"  already won by the recording side: {pop['already_won']}")
print()
print(f"SYMMETRY on all {len(pool)}")
print(f"  seats {sym['seat0']}/{sym['seat1']}, p = {sym['p']:.3f} -> "
      f"{'pass' if sym['pass'] else 'FAIL'}")
print()
print("SAMPLE AUDIT")
for r in I.audit_samples([pop, sym], pool):
    print(f"  {r['measure']:<12}{str(r['have']):>7} need {str(r['need']):<6}"
          f"{'ok' if r['ok'] else 'THIN':<5} {r['note'][:64]}")

Three things in that output are the point of the whole file.

**Effective n is 101 where nominal is 150.** Those 150 matchups were chosen to be as different from each other as a benchmark can make them, and a third of them still collapse into somebody else. That gap is the whole reason check 4 exists, and it is the best case rather than the typical one.

**`SYMMETRY` passes, and it is the only line that could have failed without your agent being involved at all.** It is the control: a quantity whose value is known in advance if nothing is wrong. Note that it passes at p = 0.07 rather than comfortably, which on a subset chosen for strength is worth watching rather than acting on.

**The audit prints `THIN` where a measure did not get the sample it declared it needs.** A harness that reports a number without saying whether the number could have detected anything is the harness that told us 72%. Declaring the requirement in advance is what makes an underpowered result readable as *not enough evidence* instead of *no effect*, and those two are written down identically by everyone who has not checked.

---

## What you are actually maximising

Start here, because it is the one result that changes what you optimise rather than how you measure it.

Take your average margin over an opponent, divide it by how much that margin wobbles from game to game, and read the answer off a normal curve. That is a prediction of your win rate against them. It sounds like a rule of thumb. It is not.

In [ ]:
# Everything below is loaded from the attached dataset, not pasted into a cell.
OC = pd.DataFrame([{k: r[k] for k in ("opponent", "n", "mu", "sigma", "z",
                                      "predicted", "observed", "se")}
                   for r in load_input("objective_calibration.json")])
FC = load_input("ladder_forecast.json")
ST = pd.DataFrame(load_input("instrument_selftest.json")["tests"])
IV = load_input("instrument_validation.json")
print(f"{len(OC)} calibration opponents, {len(ST)} self-tests, "
      f"{len(IV['instruments'])} historical instruments, "
      f"{len(FC['forecast']['band'])} forecast horizons")

In [ ]:
OC["abs_err"] = (OC.predicted - OC.observed).abs()
OC["inside"] = OC.abs_err <= 1.96 * OC.se

def flabel(s):
    """Kaggle images carry no CJK font, so such a name would render as tofu."""
    s = str(s)
    return "a top-10 team" if any("⺀" <= c <= "鿿" or
                                  "가" <= c <= "힯" for c in s) else s[:16]

fig, ax = plt.subplots(figsize=(6.4, 5.2))
ax.plot([0, 1], [0, 1], color=GREY, lw=0.8, ls=(0, (4, 3)), zorder=1,
        label="if the rule were exact")
ax.errorbar(OC.predicted, OC.observed, yerr=1.96 * OC.se, fmt="o", ms=4.5,
            mfc="white", mec=TEAL, mew=1.1, ecolor=GREY, elinewidth=0.7,
            capsize=2, capthick=0.7, zorder=3, label="one opponent, 20 games")
for _, r in OC.iterrows():
    if r.abs_err > 0.11:
        ax.annotate(flabel(r.opponent), (r.predicted, r.observed),
                    textcoords="offset points", xytext=(7, -3), fontsize=7.5,
                    color="#475569")
ax.set_xlim(0.25, 1.0); ax.set_ylim(0.25, 1.0); ax.set_aspect("equal")
ax.set_xlabel("win rate the rule predicts, from margin / wobble")
ax.set_ylabel("win rate actually observed")
ax.set_title("Wins really are margin divided by wobble", loc="left")
ax.legend(loc="upper left", fontsize=8.5)
plt.tight_layout(); plt.show()

print(f"  mean error            {OC.abs_err.mean():.3f}")
print(f"  inside the interval   {int(OC.inside.sum())}/{len(OC)} opponents")

Fourteen opponents out of fourteen land inside their interval. So "average margin divided by how much it wobbles" is not a rule of thumb here. It is the thing that predicts your win rate.

Fourteen points would not, on their own, tell a normal curve from a logistic one, and that matters because the whole rule rests on which curve it is. The reason to believe the normal is separate and structural: a single bank in this game **is** normal, measured on 45,404 of them, so a margin against a fixed opponent is a difference of normals and the probit is what the game implies rather than what happened to fit. That derivation, and the reason the same rule fails when you pool across opponents, is in [Know Your Noise](https://www.kaggle.com/code/destbreso/kaggriculture-know-your-noise).

The practical consequence is uncomfortable and it is the reason this sits at the top of part two. **When you are behind, more wobble helps you.** A change that lowers your median bank but widens its spread can raise your chance of winning, and every instinct built on "improve the average" gets that backwards. Which way to lean is not a preference, it is a signed quantity you can read off the same curve.

---

## Where will it end up? A band, not a number

One last thing, because a rating you read once is not a prediction.

A new submission starts low and climbs, then falls back with no code change at all. We wondered whether that was just us. It is not. It is what the entire field does.

In [ ]:
band = pd.DataFrame(FC["forecast"]["band"])
now = FC["now"]
fig, ax = plt.subplots(figsize=(7.6, 4.2))
ax.fill_between(band.episode, band.p10, band.p90, color=TEAL, alpha=0.16,
                label="80 % of comparable submissions")
ax.fill_between(band.episode, band.p25, band.p75, color=TEAL, alpha=0.30,
                label="the middle half")
ax.plot(band.episode, band.p50, color=TEAL, lw=1.3, label="median path")
ax.scatter([now["episodes"]], [now["score"]], s=42, facecolor="white",
           edgecolor=RED, linewidth=1.3, zorder=5, label="where ours is now")
ax.set_xlabel("episodes played")
ax.set_ylabel("ladder rating")
ax.set_title("Rating is a trajectory, so the honest answer is a band", loc="left")
ax.legend(loc="lower left", fontsize=8.5)
plt.tight_layout(); plt.show()

print(f"  built from the {FC['forecast']['peers']} submissions that sat within")
print(f"  250 rating points of ours at the same episode number")

Read across everyone's episodes, the pattern is the same for everybody:

* a submission climbs at about **+28 rating points per episode** until it peaks,
* it peaks around its **60th episode**,
* then it falls at about **3 points per episode**, indefinitely, with nobody touching the code.

That last number is why comparing your rating today against your rating last week tells you almost nothing. We spent a week ranking our own agents that way before noticing that one of them, byte-identical throughout, had fallen 182 points on its own.

*The decay is not mysterious once you look at the field as a population. Lineages sweep through it: one takes a third of every seat in two days, holds three, and is gone in five. We measured that in [the leaderboard has a fossil record](https://www.kaggle.com/code/destbreso/the-leaderboard-has-a-fossil-record). Your agent does not get worse. Its environment is replaced.*

## Does the checker itself work?

A test nobody has tested is a hypothesis. Ours is checked against four questions where the answer is already known.

In [ ]:
show = ST[["test", "expected", "got", "pass", "wins", "n", "paired_median",
           "paired_p"]].copy()
show["result"] = np.where(show["pass"], "pass", "FAIL")
display(show[["test", "expected", "got", "result", "wins", "n",
              "paired_median", "paired_p"]])

* **the same agent against itself** must come back as exactly zero. It does.
* **a change switched off** must come back as exactly zero. It does.
* **a setting we know is catastrophic** must be caught. It is, at a margin of −118,222.
* **two of our own agents whose real ladder order we know** must come out in that order. They do.

**And here is what that is still not.** The last test is a **320-point** gap between two agents. Telling apart two *close* strong agents is the question we actually ask, and nothing above tests it.

That is not a hypothetical worry. Every offline test this project ever validated was validated on nine agents spanning ladder scores **354 to 662**, and correlated beautifully across that range. One of them then confirmed the 72% that turned out to be 13%.

In [ ]:
truth, inst = IV["truth"], IV["instruments"]
RG = pd.DataFrame([{"agent": a, "ladder": truth[a], "score": inst[a]["beat_history"]}
                   for a in truth if a in inst and "beat_history" in inst[a]])

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.9),
                         gridspec_kw={"width_ratios": [1.05, 1]})
ax = axes[0]
ax.scatter(RG.score, RG.ladder, s=42, facecolor="white", edgecolor=TEAL,
           linewidth=1.1, zorder=3)
ax.set_xlabel("what the offline test said")
ax.set_ylabel("what the ladder said")
ax.set_title("Validated here: nine weak agents, 307 points apart", loc="left")

ax = axes[1]
for i, (lab, lo, hi, c) in enumerate([
        ("validated" + chr(10) + "on this range", 354, 662, TEAL),
        ("USED" + chr(10) + "on this range", 2400, 2850, RED)]):
    ax.add_patch(Rectangle((i - 0.32, lo), 0.64, hi - lo, color=c, alpha=0.80, linewidth=0))
    ax.text(i, hi + 110, lab, ha="center", fontsize=9, color=c, fontweight="bold")
    ax.text(i, (lo + hi) / 2, f"{hi - lo} pts" + chr(10) + "wide", ha="center",
            va="center", color="white", fontsize=9, fontweight="bold")
ax.set_xlim(-0.7, 1.7); ax.set_ylim(0, 3250); ax.set_xticks([])
ax.set_ylabel("ladder rating")
ax.set_title("The gap it was never asked to cross", loc="left")
plt.tight_layout(); plt.show()

Ordering agents 307 points apart is easy. The candidates you actually choose between differ by tens of points, in a band the test was never checked on.

**An offline test validated on a wide spread of weak agents has not been validated.** That sentence applies to the one in this notebook too, and saying so is the point rather than a disclaimer.

## Four ways an instrument lies

Every measurement this project has had to retract is one of four kinds. Naming them is what makes the next one findable before it costs a submission.

| | family | the question it silently answers instead | an example that cost us |
|---|---|---|---|
| **A** | **wrong object** | a quantity *adjacent* to the one you need, and correct for its own purpose | dividing by the spread between opponents instead of the spread within a pairing |
| **B** | **wrong population** | a field that no longer exists, or one you picked by having beaten it | 34 of our 40 evaluation games were games we had already won |
| **C** | **wrong estimator** | too few draws, a maximum read in sample, a statistic that is not the target | a pool of 40 has a 21% chance of noticing a real 60% effect |
| **D** | **wrong control** | no bit-identical baseline, so a change and a bug are one observation | a shadowed variable that silently rewired the farm |

**Family A is the expensive one**, and half of everything we have retracted sits in it. It is invisible to code review because the number really is correct; it is answering a different question. The only defence we have found is the one in check 4: **carry a control that tells you what the measurement looks like when nothing is happening.**

---

## When is a single number even the right object?

Check 5 gives you a comparison against one baseline. Turning many such comparisons into a ranking assumes something extra, and it is worth knowing that you assumed it: that strength is **transitive**, so that if A beats B and B beats C then A beats C. A scalar rating is a lossless summary exactly when that holds, and a lossy one when it does not.

Here is everything this project has on that question, which is not much, and the three rows disagree.

| evidence | triples | what it found |
|---|---|---|
| complete triples examined | 9 | 0 cycles found |
| schedule dynamics | - | 0 cycles, one dominator |
| solving our own payoff matrix | - | the equilibrium is MIXED, worth 0.576 over four routes |

A mixed equilibrium implies intransitivity somewhere, and nine triples cannot detect it. **Verdict: undecided**, and the harness says so rather than fitting a scalar and calling it a ranking.

Nine triples with no cycles is not evidence of transitivity; it is an absence of evidence, and those read the same in a table. Meanwhile solving the payoff matrix over our own routes gives a **mixed** equilibrium, which can only happen if some strength relation somewhere is circular. The two facts point opposite ways and the sample cannot separate them.

So the harness reports `RANKING ... THIN` and declines to produce a ranking. That is a better output than a number, because a scalar strength computed over an intransitive game is not a weak estimate of the right thing, it is a confident estimate of the wrong thing.

---

## Every one of these has a name

We did not invent any of these mistakes. Each one has a name, a literature and usually a canonical example from some other field, and the names are worth knowing because **you recognise a failure much faster once you can call it something**.

Here is the whole set, with what each cost us.

| what we did | its name | what it cost us |
|---|---|---|
| scored candidates against near-duplicate opponents | **design effect** | 40 opponents were 2 strategies |
| tested against games we had already won | **survivorship bias** | the test could not show us losing |
| kept the best of 259 settings and quoted its score | **the winner's curse** | 20 points, same agent |
| validated on agents 307 points apart, then used it on agents 30 apart | **range restriction** | 72% offline, 13% live |
| compared this week's rating against last week's | **non-stationarity** | 182 points of drift read as quality |
| compared win rates of agents who faced different opponents | **confounding** | 59% against 22%, zero shared opponents |
| took one evaluation as evidence about an objective | **the law of small numbers** | we invented an objective we already had |
| assumed a single number can rank everyone | **intransitivity** | still unmeasured, see below |
| played both agents on the same seeds | **common random numbers** | the one we got right, and it is worth an order of magnitude |

One more, which is about the game rather than about the measurement, and which is the reason the pool question in check 4 is not a technicality.

## If you take three things

1. **Play the identical games.** Same seed, same seat, same opponent, then difference them. It is the cheapest real improvement available and it is worth more than an order of magnitude in games.
2. **Count your opponents by what they do.** Ours were two strategies wearing forty names, and every candidate we selected was ranked against that.
3. **Tune on one half, report the other.** Twenty points of our best number was the act of choosing it.

And one habit underneath all three: **carry a control that tells you what the measurement looks like when nothing is happening.** A bit-identical agent, a null comparison, an interval that contains zero. Nearly every wrong number in this notebook was wrong in a way that looked entirely reasonable, and the control is what made it visible.

---

*Everything here is from our own runs on this competition, failures included. If you fork it and run the checks on your own agent, the numbers worth your attention are the ones that surprise you.*

---

## If you want the rest of the series

These are the measurements this notebook keeps leaning on. Each answers one question about the field rather than about any single agent.

- [**The leaderboard is a habitat gradient**](https://www.kaggle.com/code/destbreso/the-leaderboard-is-a-habitat-gradient). What you meet at one rating band is not what you meet at another, which is why a fixed pool of opponents measures a fixed place.
- [**Everyone is playing the same opening**](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening). Unrelated teams emit identical actions, turn for turn, deep into the game. This is why check 4 exists.
- [**Mutants at the top**](https://www.kaggle.com/code/destbreso/mutants-at-the-top-genetic-potential-of-a-replay). Three of the top ten hold one farm plan across every episode and vary only what they sell, so you can locate which part of a strong agent is safe to change.
- [**The leaderboard has a fossil record**](https://www.kaggle.com/code/destbreso/the-leaderboard-has-a-fossil-record). The field turns over by sweeps with a half life of 2.1 days, measured. This is why a rating from last week compares to nothing.
- [**What kind of optimisation is this?**](https://www.kaggle.com/code/destbreso/kaggriculture-what-kind-of-optimisation-is-this). Once you freeze the farm plan, what is left is a solvable problem with a known price curve, and it explains why we search parameters rather than train a policy.

---

<details>
<summary>&nbsp;<b>&#9656;&nbsp; THE FULL BIBLIOGRAPHY: every source, in one place</b>&nbsp;&nbsp;<i>(click to open)</i></summary>
<div style="padding: 0.4em 0 0.2em 1.1em;">

<p><b>Playing the identical games</b></p>
<ul>
<li>Law, A. M. & Kelton, W. D. <i>Simulation Modeling and Analysis.</i> McGraw-Hill.</li>
<li>Glasserman, P. & Yao, D. D. (1992). Some guidelines and guarantees for common random numbers. <i>Management Science</i> 38(6), 884-908.</li>
</ul>
<p><b>Counting your opponents</b></p>
<ul>
<li>Kish, L. (1965). <i>Survey Sampling.</i> Wiley.</li>
</ul>
<p><b>Choosing the best of many, and paying for it</b></p>
<ul>
<li>Kim, S.-H. & Nelson, B. L. (2006). Selecting the best system. <i>Handbooks in OR & MS</i>, vol. 13.</li>
<li>Chen, C.-H., Lin, J., Yücesan, E. & Chick, S. E. (2000). Simulation budget allocation for further enhancing the efficiency of ordinal optimization. <i>Discrete Event Dynamic Systems</i> 10, 251-270.</li>
<li>Karnin, Z., Koren, T. & Somekh, O. (2013). Almost optimal exploration in multi-armed bandits. <i>ICML.</i></li>
</ul>
<p><b>Rating players from pairwise results</b></p>
<ul>
<li>Bradley, R. A. & Terry, M. E. (1952). Rank analysis of incomplete block designs. <i>Biometrika</i> 39(3/4), 324-345.</li>
<li>Hunter, D. R. (2004). MM algorithms for generalized Bradley-Terry models. <i>Annals of Statistics</i> 32(1), 384-406.</li>
<li>Glickman, M. E. (1999). Parameter estimation in large dynamic paired comparison experiments. <i>JRSS-C</i> 48(3), 377-394.</li>
<li>Herbrich, R., Minka, T. & Graepel, T. (2007). TrueSkill: a Bayesian skill rating system. <i>NeurIPS.</i></li>
<li>Coulom, R. (2008). Whole-History Rating. <i>Computers and Games.</i></li>
</ul>
<p><b>When one number is the wrong object</b></p>
<ul>
<li>Balduzzi, D., Tuyls, K., Perolat, J. & Graepel, T. (2018). Re-evaluating evaluation. <i>NeurIPS 31.</i></li>
<li>Omidshafiei, S. et al. (2019). α-Rank: multi-agent evaluation by evolution. <i>Scientific Reports</i> 9, 9937.</li>
<li>Czarnecki, W. M. et al. (2020). Real world games look like spinning tops. <i>NeurIPS 33.</i></li>
</ul>
<p><b>What you are actually maximising in a two-player duel</b></p>
<ul>
<li>Immorlica, N., Kalai, A. T., Lucier, B., Moitra, A., Postlewaite, A. & Tennenholtz, M. (2011). Dueling Algorithms. <i>STOC 2011</i>, 215-224.</li>
</ul>
<p><b>Public work in this competition that this notebook uses or argues with</b></p>
<ul>
<li>Rayk Kretzschmar, <a href="https://www.kaggle.com/code/raykkretzschmar/kaggriculture-rank-your-agent">rank your agent</a>.</li>
<li>Kaito Fukami, <a href="https://www.kaggle.com/code/kaitofukami/41-49-future-unseen-v19-replication-to-control">41/49 Future+Unseen</a>.</li>
<li>Busya PRIME, <a href="https://www.kaggle.com/code/busyaprime/what-actually-wins-on-the-kaggriculture-ladder">What actually wins on the Kaggriculture ladder</a>.</li>
<li>Furina, <a href="https://www.kaggle.com/code/cjlcjlcjl/kaggriculture-what-the-top-farms-do-a-live-meta">What the top farms do</a>.</li>
</ul>
<p><i>Volumes and page numbers are written from memory and are worth checking before quoting; the titles and years are the load-bearing part.</i></p>

</div>
</details>

---